# Solutions — Bakehouse Deep Analytics (Hard 02)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

transactions = spark.table("samples.bakehouse.sales_transactions")
customers    = spark.table("samples.bakehouse.sales_customers")
franchises   = spark.table("samples.bakehouse.sales_franchises")
suppliers    = spark.table("samples.bakehouse.sales_suppliers")
reviews      = spark.table("samples.bakehouse.media_customer_reviews")
gold_reviews = spark.table("samples.bakehouse.media_gold_reviews_chunked")


## Solution 1 — Full Transaction Enrichment

In [ ]:
result_1 = (
    transactions
    .join(customers.select("customerID", F.col("country").alias("customer_country")), on="customerID")
    .join(
        franchises.select("franchiseID", F.col("name").alias("franchiseName"),
                          F.col("country").alias("franchise_country"), F.col("size").alias("franchise_size")),
        on="franchiseID"
    )
    .select(
        "transactionID", "dateTime", "product", "quantity", "totalPrice",
        "customerID", "customer_country", "franchiseName", "franchise_country", "franchise_size"
    )
)


## Solution 2 — Customer Lifetime Value

In [ ]:
cust_stats = (
    transactions
    .groupBy("customerID")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("total_spend"),
        F.count("*").alias("transaction_count"),
        F.round(F.avg("totalPrice"), 2).alias("avg_order_value"),
        F.min("dateTime").alias("first_purchase"),
        F.max("dateTime").alias("last_purchase"),
    )
    .withColumn("tenure_days",
        F.greatest(F.datediff("last_purchase", "first_purchase"), F.lit(1))
    )
    .withColumn("annualised_ltv",
        F.round(F.col("total_spend") / F.col("tenure_days") * 365, 2)
    )
)

result_2 = (
    cust_stats
    .join(
        customers.select("customerID", F.col("firstName").alias("first_name"), F.col("lastName").alias("last_name")),
        on="customerID"
    )
    .select("customerID", "first_name", "last_name", "total_spend", "transaction_count",
            "avg_order_value", "tenure_days", "annualised_ltv")
    .orderBy(F.col("annualised_ltv").desc())
)


## Solution 3 — Revenue Cohort Analysis

In [ ]:
# First purchase month per customer
first_purchase = (
    transactions
    .groupBy("customerID")
    .agg(F.min("dateTime").alias("first_dt"))
    .withColumn("cohort_month", F.date_trunc("month", "first_dt"))
)

# Cohort size
cohort_size = (
    first_purchase
    .groupBy("cohort_month")
    .agg(F.countDistinct("customerID").alias("cohort_size"))
)

# Join transactions with cohort info
enriched = (
    transactions
    .join(first_purchase, on="customerID")
    .withColumn("tx_month", F.date_trunc("month", "dateTime"))
    .withColumn("months_since_first",
        F.cast(F.months_between("tx_month", "cohort_month"), "int")
    )
    .filter(F.col("months_since_first") >= 0)
)

revenue_by_cohort = (
    enriched
    .groupBy("cohort_month", "months_since_first")
    .agg(F.round(F.sum("totalPrice"), 2).alias("revenue"))
)

result_3 = (
    revenue_by_cohort
    .join(cohort_size, on="cohort_month")
    .orderBy("cohort_month", "months_since_first")
)


## Solution 4 — Discount Detection

In [ ]:
result_4 = (
    transactions
    .withColumn("expected_price", F.round(F.col("unitPrice") * F.col("quantity"), 2))
    .filter(F.col("totalPrice") < F.col("expected_price"))
    .withColumn("discount_amount", F.round(F.col("expected_price") - F.col("totalPrice"), 2))
    .withColumn("discount_pct",    F.round((F.col("expected_price") - F.col("totalPrice")) / F.col("expected_price") * 100, 2))
    .select("transactionID", "customerID", "franchiseID", "product", "unitPrice", "quantity",
            "totalPrice", "expected_price", "discount_amount", "discount_pct")
    .orderBy(F.col("discount_pct").desc())
)


## Solution 5 — Franchise Performance Scorecard

In [ ]:
rev_stats = (
    transactions
    .groupBy("franchiseID")
    .agg(
        F.round(F.sum("totalPrice"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count"),
        F.countDistinct("customerID").alias("unique_customers"),
    )
)

rev_score = (
    reviews
    .groupBy("franchiseID")
    .agg(F.round(F.avg("reviewScore"), 2).alias("avg_review_score"))
)

w = Window.orderBy("total_revenue_rank", "avg_rev_rank")
rev_w = Window.orderBy(F.col("total_revenue").desc())
rat_w = Window.orderBy(F.col("avg_review_score").desc())

result_5 = (
    rev_stats
    .join(rev_score, on="franchiseID", how="left")
    .join(franchises.select("franchiseID", F.col("name").alias("franchiseName")), on="franchiseID")
    .withColumn("total_revenue_rank", F.rank().over(rev_w))
    .withColumn("avg_rev_rank",       F.rank().over(rat_w))
    .withColumn("composite_rank",     F.rank().over(Window.orderBy(
        (F.col("total_revenue_rank") + F.coalesce(F.col("avg_rev_rank"), F.lit(9999))).asc()
    )))
    .select("franchiseID", "franchiseName", "total_revenue", "transaction_count",
            "unique_customers", "avg_review_score", "composite_rank")
    .orderBy("composite_rank")
)


## Solution 6 — Product Affinity — Frequently Bought Together

In [ ]:
result_6 = (
    transactions.alias("a")
    .join(transactions.alias("b"), on="transactionID")
    .filter(F.col("a.product") < F.col("b.product"))   # deduplicate pairs
    .groupBy(
        F.col("a.product").alias("product_a"),
        F.col("b.product").alias("product_b"),
    )
    .agg(F.count("*").alias("co_occurrence_count"))
    .orderBy(F.col("co_occurrence_count").desc())
)


## Solution 7 — PII Data Masking

In [ ]:
import re as _re

result_7 = (
    transactions
    .join(customers,  on="customerID")
    .join(franchises.select("franchiseID", F.col("name").alias("franchiseName")), on="franchiseID")
    .withColumn("firstName",    F.lit("***"))
    .withColumn("lastName",     F.lit("***"))
    .withColumn("emailAddress",
        F.concat(
            F.substring(F.col("emailAddress"), 1, 1),
            F.lit("*******@"),
            F.regexp_extract(F.col("emailAddress"), r"@(.+)$", 1)
        )
    )
    .withColumn("phoneNumber",
        F.concat(F.lit("***-***-"), F.substring(F.regexp_replace("phoneNumber", r"[^0-9]", ""), -3, 3))
    )
    .withColumn("maskedCard",
        F.concat(F.lit("****-****-****-"), F.substring(F.cast("cardNumber", "string"), -4, 4))
    )
    .select(
        F.col("dateTime").alias("dateTime"),
        "franchiseName", "firstName", "lastName",
        "emailAddress", "phoneNumber", "maskedCard",
        "product", "quantity",
        F.col("totalPrice").alias("totalPrice"),
        "paymentMethod"
    )
)


## Solution 8 — Data Quality Report

In [ ]:
tables = {
    "sales_transactions": (transactions, "transactionID"),
    "sales_customers":    (customers,    "customerID"),
    "sales_franchises":   (franchises,   "franchiseID"),
    "sales_suppliers":    (suppliers,    "supplierID"),
}

rows = []
for tname, (df, pk) in tables.items():
    row_count   = df.count()
    null_cols   = [c for c in df.columns if df.filter(F.col(c).isNull()).count() > 0]
    null_str    = ",".join(null_cols) if null_cols else ""
    dup_pk      = row_count - df.select(pk).distinct().count()
    rows.append((tname, row_count, null_str, dup_pk))

result_8 = spark.createDataFrame(rows,
    ["table_name", "row_count", "null_columns", "duplicate_pk_count"])
